In [32]:
import os.path as osp
import random

import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.utils.data as data
import torchvision
from cv2.gapi import kernel
from sympy.abc import lamda

USE_MPS=torch.backends.mps.is_available()
DEVICE=torch.device('mps' if USE_MPS else 'cpu')

In [2]:
seed = 1000
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [3]:
# 이미지 정보
base_dir = '../../data/VOCdevkit/VOC2012/'
img_data_template = osp.join(base_dir, 'JPEGImages', '%s.jpg')
annopath_template = osp.join(base_dir, 'Annotations', '%s.xml')

In [4]:
# 학습을 위한 이미지 지정 데이터
tr_ids = osp.join(base_dir, 'ImageSets/Main/train.txt')
val_ids = osp.join(base_dir, 'ImageSets/Main/val.txt')

In [5]:
tr_img_list = []
tr_anno_list = []

for i in open(tr_ids):
    tr_img_list.append(img_data_template % i.strip())
    tr_anno_list.append(annopath_template % i.strip())

In [6]:
tr_img_list

['../../data/VOCdevkit/VOC2012/JPEGImages/2008_000008.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000015.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000019.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000023.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000028.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000033.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000036.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000037.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000041.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000045.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000053.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000060.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000066.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000070.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000074.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000085.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImage

In [7]:
val_img_list = []
val_anno_list = []

for i in open(val_ids):
    val_img_list.append(img_data_template % i.strip())
    val_anno_list.append(annopath_template % i.strip())

In [8]:
val_img_list

['../../data/VOCdevkit/VOC2012/JPEGImages/2008_000002.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000003.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000007.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000009.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000016.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000021.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000026.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000027.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000032.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000034.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000042.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000043.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000050.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000051.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000052.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImages/2008_000054.jpg',
 '../../data/VOCdevkit/VOC2012/JPEGImage

In [9]:
# 클래스 화 하기
def make_datapath_list(rootpath):
    img_data_template = osp.join(rootpath, 'JPEGImages', '%s.jpg')
    annopath_template = osp.join(rootpath, 'Annotations', '%s.xml')

    tr_ids = osp.join(rootpath, 'ImageSets/Main/train.txt')
    val_ids = osp.join(rootpath, 'ImageSets/Main/val.txt')

    tr_img_list = []
    tr_anno_list = []

    for i in open(tr_ids):
        tr_img_list.append(img_data_template % i.strip())
        tr_anno_list.append(annopath_template % i.strip())

    val_img_list = []
    val_anno_list = []

    for i in open(val_ids):
        val_img_list.append(img_data_template % i.strip())
        val_anno_list.append(annopath_template % i.strip())

    return tr_img_list, tr_anno_list, val_img_list, val_anno_list

In [10]:
base_dir = '../../data/VOCdevkit/VOC2012/'
tr_img_list, tr_anno_list, val_img_list, val_anno_list = make_datapath_list(base_dir)

In [11]:
ck_xml = tr_anno_list[0]

In [12]:
ck_xml

'../../data/VOCdevkit/VOC2012/Annotations/2008_000008.xml'

In [13]:
l = ['a', 'b']
l.index('b')

1

In [ ]:
xml = ET.parse(ck_xml).getroot()
width = 500
height = 442
class_name_list = {key:10}

for i in xml.iter('object'):
    difficult = int(i.find('difficult').text)
    if difficult == 1:
        continue
    bndbox = []
    name = i.find('name').text.lower().strip()
    bbox = i.find('bndbox')
    pts = ['xmin', 'ymin', 'xmax', 'ymax']
    for pt in pts:
        cur_pixel = int(bbox.find(pt).text)-1

        if pt == 'xmin' or pt == 'xmax':
            cur_pixel /= width
        else:
            cur_pixel /= height
        bndbox.append(cur_pixel)
    label_index = class_name_list.index(name)
    bndbox.append(label_index)

In [15]:
class Anno_xml2list:
    def __init__(self,classes):
        self.classes=classes
    def __call__(self,xml_path,width,height):
        ret=[]
        xml=ET.parse(ck_xml).getroot()
        for i in xml.iter('object'):
            difficult=int(i.find('difficult').text)
            if difficult==1:
                continue
            bndbox=[]
            name=i.find('name').text.lower().strip()
            bbox=i.find('bndbox')
            pts=['xmin','ymin','xmax','ymax']
            for pt in pts:
                cur_pixel=int(bbox.find(pt).text)-1

                if pt=='xmin' or pt=='xmax':
                    cur_pixel /= width
                else:
                    cur_pixel /= height
                bndbox.append(cur_pixel)
            label_index=self.classes.index(name)
            bndbox.append(label_index)
            ret.append(bndbox)
        return np.array(ret)

In [16]:
from pathlib import Path
f_data = Path(base_dir+'ImageSets/Main/')
set_v = set()
for i in f_data.iterdir():
    if '_' in i.name:
        set_v.add(i.name.split('_')[0])
sorted(list(set_v))

['aeroplane',
 'bicycle',
 'bird',
 'boat',
 'bottle',
 'bus',
 'car',
 'cat',
 'chair',
 'cow',
 'diningtable',
 'dog',
 'horse',
 'motorbike',
 'person',
 'pottedplant',
 'sheep',
 'sofa',
 'train',
 'tvmonitor']

In [17]:
voc_classes=['aeroplane', 'bicycle', 'bird', 'boat',
'bottle', 'bus', 'car', 'cat', 'chair',
'cow', 'diningtable', 'dog', 'horse',
'motorbike', 'person', 'pottedplant',
'sheep', 'sofa', 'train', 'tvmonitor']

transform_anno=Anno_xml2list(voc_classes)
idx=123
img_path=tr_img_list[idx]
img=cv2.imread(img_path)
h,w,c=img.shape
ann_img_tr=transform_anno(tr_anno_list[idx],w,h)
ann_img_tr

array([[ 0.104     ,  0.22933333,  0.94      ,  1.11733333, 12.        ],
       [ 0.314     ,  0.11466667,  0.576     ,  0.44266667, 14.        ]])

In [18]:
class Compose:
    def __init__(self, transforms):
        self.transforms=transforms
    def __call__(self, img, boxes = None, label = None):
        for t in self.transforms:
            img, boxes, label = t(img, boxes, label)
        return img, boxes, label

class ConvertFromInts:
    def __call__(self, img, boxes = None, label = None):
        return img.astype(np.float32), boxes, label

class ToAbsoluteCoords:
    def __call__(self, img, boxes = None, label = None):
        h, w, c = img.shape
        boxes[:, 0] *= w
        boxes[:, 2] *= w
        boxes[:, 1] *= h
        boxes[:, 3] *= h
        return img, boxes, label

class ToPercentCoords:
    def __call__(self, img, boxes = None, label = None):
        h, w, c = img.shape
        boxes[:, 0] /= w
        boxes[:, 2] /= w
        boxes[:, 1] /= h
        boxes[:, 3] /= h
        return img, boxes, label

class Resize:
    def __init__(self, size = 300):
        self.size = size
    def __call__(self, img, boxes = None, label = None):
        cv2.resize(img, (self.size, self.size))
        return img, boxes, label

class ConvertColor:
    def __init__(self, c = 'BGR', tr = 'HSV'):
        self.transform = tr
        self.current = c
    def __call__(self, img, boxes = None, label = None):
        if self.current == 'BGR' and self.transform == 'HSV':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        elif self.current == 'HSV' and self.transform == 'BGR':
            img = cv2.cvtColor(img, cv2.COLOR_HSV2BGR)
        else:
            raise NotImplementedError
        return img, boxes, label

def interset(box_a, box_b):
    max_xy = np.minimum(box_a[:,2:], box_b[2:])
    min_xy = np.maximum(box_a[:,:2], box_b[2:2])
    inter = np.clip((max_xy - min_xy), 0, np.inf)

def jacquard_numpy(box_a, box_b):
    inter = (box_a, box_b)
    ares_a = (box_a[:,2]-box_a[:,0])*(box_a[:,3]-box_a[:,1])
    ares_b = (box_b[:,2]-box_b[:,0])*(box_b[:,3]-box_b[:,1])
    union = ares_a+ares_b-inter
    return inter/union

class RandomContrast:
    def __init__(self, lower = .5, upper = .5):
        self.lower = lower
        self.upper = upper
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            alpha = random.uniform(self.lower, self.upper)
            img += alpha
        return img, boxes, label

class RandomSaturation:
    def __init__(self, lower = .5, upper = .5):
        self.lower = lower
        self.upper = upper
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            img[:,:,1] *= random.uniform(self.lower, self.upper)
        return img, boxes, label

class RandomHue:
    def __init__(self, delta = 18.0):
        assert .0 <= delta >= 360.0
        self.delta = delta
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            img[:,:,0] += random.uniform(-self.delta, self.delta)
            img[:,:,][img[:,:,0] > 360.0] -= 360.0
            img[:,:,][img[:,:,0] < .0] += 360.0
        return img, boxes, label

class RandomBrightness:
    def __init__(self, delta = 18.0):
        assert .0 <= delta >= 255.0
        self.delta = delta
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            alpha = random.uniform(-self.delta, self.delta)
            img += alpha
        return img, boxes, label

class RandomLightNosie:
    def __init__(self):
        self.perms = ((0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0))
    def __call__(self, img, boxes = None, label = None):
        if random.randint(2):
            swap = self.perms[random.randint(len(self.perms))]
            img = img[:,:,swap]
        return img, boxes, label

class RandomSampleCrop:
    def __init__(self):
        self.sample_option = ((None, None),
                              (.1, None),
                              (.3, None),
                              (.7, None),
                              (.9,None), None)
    def __call__(self, img, boxes = None, label = None):
        h, w, c = img.shape
        while True:
            idx = np.random.randint(len(self.sample_option))
            mode = self.sample_option[idx]
            if mode is None:
                return img, boxes, label
            min_iou, max_iou = mode
            if min_iou is None:
                min_iou = float('-inf')
            if max_iou is None:
                max_iou = float('-inf')
            for _ in range(50):
                current_img = img
                tr_w = random.uniform(.3*w,w)
                tr_h = random.uniform(.3*h,h)
                if tr_h/tr_w<.5 or tr_h/tr_w>2:
                    continue
                left = random.uniform(w-tr_w)
                top = random.uniform(h-tr_h)

                rect = np.array([int(left), int(top), int(left+tr_w), int(top+tr_h)])
                overlap =jacquard_numpy(boxes, rect)
                if overlap.min()<min_iou and overlap.max()>max_iou:
                    continue
                current_img = current_img[rect[1]:rect[3], rect[0]:rect[2],:]
                centers = (boces[:,:2]+boxes[:,:2])/2.0

                m1 = (rect[0]<centers[:,0])*(rect[1]<centers[:,1])
                m2 = (rect[2]<centers[:,0])*(rect[3]<centers[:,1])
                mask = m1 * m2

                if not mask.any():
                    continue

                current_boxes = boxes[mask, :].copy()
                current_label = label[mask]

                current_boxes[:, :2] = np.maximum(current_boxes[:, :2], rect[:2])
                current_boxes[:, :2] -= rect[:2]
                current_boxes[:, 2:] = np.minimum(current_boxes[:, 2:], rect[2:])
                current_boxes[:, 2:] -= rect[2:]
                return current_img, current_boxes, current_label

class PhotometricDistort:
    def __init__(self):
        self.pd = [
            RandomContrast(),
            ConvertColor(),
            RandomSaturation(),
            RandomHue(),
            ConvertColor('HSV', 'BGR'),
            RandomContrast()
        ],
        self.rand_brightness = RandomBrightness()
        self.rand_lightness_noise = RandomLightNosie()

    def __call__(self, img, boxes = None, label = None):
        im = img.copy()
        im, boxes, label = self.rand_brightness(im, boxes, label)
        if random.randint(2):
            distort = Compose(self.pd[:-1])
        else:
            distort = Compose(self.pd[1:])
        im, boxes, label = distort(im, boxes, label)
        return self.rand_lightness_noise(im, boxes, label)

def main():
    pass

if __name__ == '__main__':
    main()

In [19]:
from utils.data_a import *

In [20]:
#데이터 로더 구조 정의
class Make_dataset_Transform:
    def __init__(self,input_size,color_mean):
        self.base_transform={
            'train':Compose([
                ConvertFromInts(), # int를 float으로 변환
                ToAbsoluteCoords(), # 어노테이션 데이터의 규격화
                PhotometricDistort(), # 랜덤한 색조 변경
                RandomSampleCrop(), # 이미지의 랜덤 샘플화(이미지 증강구조 추가 가능)
                ToPercentCoords(), # 데이터 패턴 규격화(0~1)
                Resize(input_size), # 이미지 크기 변경
                SubtractMeans(color_mean) # 평균값 차연산
            ]),
            'val':Compose([
                ConvertFromInts(), # int를 float으로 변환
                Resize(input_size), # 이미지 크기 변경
                SubtractMeans(color_mean) # 평균값 차연산
        ])
        }
    def __call__(self, img, phase, boxes, label):
        return self.base_transform[phase](img, boxes, label)

In [21]:
idx = 0
img_path = tr_img_list[idx]
img = cv2.imread(img_path)
h, w, c = img.shape

transform_anno = Anno_xml2list(voc_classes)
anno_list = transform_anno(tr_anno_list[idx], w, h)

In [22]:
color_mean = (104, 117, 123) # BGR 평균
input_size = 300

tr = Make_dataset_Transform(input_size, color_mean)

# 모델

In [23]:
import os
import urllib.request

In [24]:
weights_dir = './weights/'

if not os.path.exists(weights_dir):
    os.makedirs(weights_dir)

url = "https://s3.amazonaws.com/amdegroot-models/vgg16_reducedfc.pth"
t_path = os.path.join(weights_dir, 'vgg16_reducedfc.pth')
if not os.path.exists(t_path):
    urllib.request.urlretrieve(url, t_path)

In [25]:
url = "https://s3.amazonaws.com/amdegroot-models/ssd300_mAP_77.43_v2.pth"
t_path = os.path.join(weights_dir, 'ssd300.pth')
if not os.path.exists(t_path):
    urllib.request.urlretrieve(url, t_path)

In [30]:
from math import sqrt
from itertools import product

import pandas as pd
import torch
# from torch.autograd import Function
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

In [37]:
def make_vgg():
    layers = []
    in_c = 3
    # VGG16 모듈 합성곱 층, max pooling 채널 수 정의
    cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'MC', 512, 512, 512, 'M', 512, 512, 512]
    for i in cfg:
        if i == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        elif i == 'MC':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)]
        else:
            conv2d = nn.Conv2d(in_c, i, kernel_size=3, padding=1)
            layers += [conv2d, nn.ReLU(inplace=True)]
            in_c = i
    pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
    conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6)
    conv7 = nn.Conv2d(1024, 1024, kernel_size=1)
    layers += [pool5, conv6, nn.ReLU(inplace=True), conv7, nn.ReLU(inplace=True)]
    return nn.ModuleList(layers)

vgg_m_test = make_vgg()
vgg_m_test

ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [35]:
def make_extras():
    layers = []
    in_c = 1024

    cfg = [256, 512, 128, 256, 128, 256, 128, 256]
    layers += [nn.Conv2d(in_c, cfg[0], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[0], cfg[1], kernel_size=(3), stride=2, padding=1)]
    layers += [nn.Conv2d(cfg[1], cfg[2], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[2], cfg[3], kernel_size=(3), stride=2, padding=1)]
    layers += [nn.Conv2d(cfg[3], cfg[4], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[4], cfg[5], kernel_size=(3))]
    layers += [nn.Conv2d(cfg[5], cfg[6], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[6], cfg[7], kernel_size=(3))]
    return nn.ModuleList(layers)
make_extras()

ModuleList(
  (0): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
  (1): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (2): Conv2d(512, 128, kernel_size=(1, 1), stride=(1, 1))
  (3): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (4): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1))
  (5): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1))
  (6): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1))
  (7): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1))
)

In [38]:
def make_loc_conf(class_n = 21, bbox_aspect_num = [4, 6, 6, 6, 4, 4]):
        loc_layers = []
        conf_layers = []

        loc_layers += [nn.Conv2d(512, bbox_aspect_num[0]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(512, bbox_aspect_num[0]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(1024, bbox_aspect_num[1]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(1024, bbox_aspect_num[1]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(512, bbox_aspect_num[2]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(512, bbox_aspect_num[2]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(256, bbox_aspect_num[3]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(256, bbox_aspect_num[3]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(256, bbox_aspect_num[4]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(256, bbox_aspect_num[4]*class_n, kernel_size=3, padding=1)]

        loc_layers += [nn.Conv2d(256, bbox_aspect_num[5]*4, kernel_size=3, padding=1)]
        conf_layers += [nn.Conv2d(256, bbox_aspect_num[5]*class_n, kernel_size=3, padding=1)]

        return nn.ModuleList(loc_layers), nn.ModuleList(conf_layers)

make_loc_conf()

(ModuleList(
   (0): Conv2d(512, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (1): Conv2d(1024, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (2): Conv2d(512, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (3): Conv2d(256, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (4-5): 2 x Conv2d(256, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
 ),
 ModuleList(
   (0): Conv2d(512, 84, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (1): Conv2d(1024, 126, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (2): Conv2d(512, 126, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (3): Conv2d(256, 126, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (4-5): 2 x Conv2d(256, 84, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
 ))

In [39]:
class L2Norm(nn.Module):
    def __init__(self, input_c = 512, scale = 20):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(input_c))
        self.scale = scale
        self.reset_parameter()
        self.eps = 1e-10

    def reset_parameter(self):
        init.constant_(self.weight, self.scale)

    def forward(self, x):
        norm = x.pow(2).sum(dim = 1, keepdim = True).sqrt()+self.eps
        x = torch.div(x, norm)
        weights = self.weight.unsqueeze(0).unsqueeze(2).unsqueeze(3).expand_as(x)
        out = weights*x
        return out

In [ ]:
ssd_cfg = {
    'num_classes': 21,  # 배경 클래스를 포함한 총 클래스 수
    'input_size': 300,  # 화상의 입력 크기
    'bbox_aspect_num': [4, 6, 6, 6, 4, 4],  # 출력할 Box 화면비의 종류
    'feature_maps': [38, 19, 10, 5, 3, 1],  # 각 source의 화상 크기
    'steps': [8, 16, 32, 64, 100, 300],  # DBOX의 크기를 정한다
    'min_sizes': [30, 60, 111, 162, 213, 264],  # DBOX의 크기를 정한다
    'max_sizes': [60, 111, 162, 213, 264, 315],  # DBOX의 크기를 정한다
    'aspect_ratios': [[2], [2, 3], [2, 3], [2, 3], [2], [2]],
}

In [40]:
class DBox:
    def __init__(self, cfg):
        super().__init__()
        self.img_size = cfg['input_size']
        self.feature_maps = cfg['feature_maps']
        self.num_priors = len(cfg['feature_maps'])
        self.steps = cfg['steps']
        self.min_sizes = cfg['min_sizes']
        self.max_sizes = cfg['max_sizes']
        self.aspect_ratios = cfg['aspect_ratios']

    def make_dbox_list(self):
        mean = []
        for k,f in enumerate(self.feature_maps):
            for i,j in product(range(f), repeat = 2):
                f_k = self.img_size/self.steps[k]

                cx = (j+.5)/f_k
                cy = (i+.5)/f_k

                s_k = self.min_sizes[k]/self.img_size
                mean += [cx, cy, s_k, s_k]

                s_k_prime = sqrt(s_k*(self.max_sizes[k]/self.img_size))
                mean += [cx, cy, s_k_prime, s_k_prime]

                for a in self.aspect_ratios[k]:
                    mean += [cx, cy, s_k*sqrt(a), s_k/sqrt(a)]
                    mean += [cx, cy, s_k/sqrt(a), s_k*sqrt(a)]
        output = torch.Tensor(mean).view(-1, 4)
        output.clamp_(max=1, min=0)
        return output

In [42]:
dbox = DBox(ssd_cfg)
pd.DataFrame(dbox.make_dbox_list())

,0,1,2,3
0,0.013333,0.013333,0.100000,0.100000
1,0.013333,0.013333,0.141421,0.141421
2,0.013333,0.013333,0.141421,0.070711
3,0.013333,0.013333,0.070711,0.141421
4,0.040000,0.013333,0.100000,0.100000
...,...,...,...,...
8727,0.833333,0.833333,0.502046,1.000000
8728,0.500000,0.500000,0.880000,0.880000
8729,0.500000,0.500000,0.961249,0.961249
8730,0.500000,0.500000,1.000000,0.622254


In [ ]:
class SSD(nn.Module):